In [0]:
from pyspark.sql.functions import *

data = [
    (1, "E", "Marketing", 6000),
    (2, "E", "Engineering", 9000),
    (3, "E", "Marketing", 7500),
    (4, "E", "Engineering", 12000),
    (5, "E", "Marketing", 8000),
    (6, "E", "Engineering", 11000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
df_agg = df.agg(max(when(col("department") == "Engineering", col("salary"))).alias("max_salary_engg"),
                max(when(col("department") == "Marketing", col("salary"))).alias("max_salary_Mark"))

df_result = df_agg.select(col("max_salary_engg")-col("max_salary_Mark")).alias("sub")

display(df_result)

In [0]:
%sql
select current_catalog();
select current_schema();

In [0]:
%sql
CREATE or replace TABLE survey_log (
  id INT,
  action STRING,
  question_id INT,
  answer_id INT,
  q_num INT,
  timestamp INT
) USING DELTA;

INSERT INTO survey_log VALUES
(5, 'show', 285, NULL, 1, 123),
(5, 'answer', 285, 124124, 1, 124),
(5, 'show', 369, NULL, 2, 125),
(5, 'skip', 369, NULL, 2, 126),
(6, 'show', 285, NULL, 1, 130),
(6, 'show', 369, NULL, 1, 131),
(7, 'show', 285, NULL, 1, 140),
(7, 'show', 369, NULL, 2, 141),
(7, 'skip', 285, NULL, 1, 142);

In [0]:
df = spark.read.table("survey_log")
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
df1 = df.withColumn("cnt", sum(when(col("action") == "answer", 1).otherwise(0)).over(Window.partitionBy("question_id"))/
        sum(when(col("action") == "show", 1).otherwise(0)).over(Window.partitionBy("question_id")))
df2 = df1.orderBy(col("cnt").desc()).limit(1)
display(df2)

In [0]:
data = [
    (1,20,"2019-08-14"),
    (2,50,"2019-08-14"),
    (1,30,"2019-08-15"),
    (1,35,"2019-08-16"),
    (2,65,"2019-08-17"),
    (3,20,"2019-08-18")
]
columns = ["product_id","new_price","change_date"]
df = spark.createDataFrame(data, columns)
df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
df2 = df.filter(to_date("change_date") <= "2019-08-16").orderBy("product_id")
df3 = df2.withColumn("rn", row_number().over(Window.partitionBy("product_id").orderBy(col("change_date").desc())))
df4 = df3.filter(col("rn") == 1)
display(df4)



In [0]:
df_dis = df.select("product_id").distinct()
df_join = df_dis.join(df4,"product_id","left")

display(df_join.select(df_dis.product_id,coalesce(col("new_price"),lit("10")).alias("price")))

In [0]:
# python
from pyspark.sql.functions import *

data = [
    (1, "Alice", "IT", 90000),
    (2, "Bob", "IT", 120000),
    (3, "Charlie", "HR", 70000),
    (4, "David", "IT", 60000),
    (5, "Eve", "HR", 75000),
    (6, "Frank", "Finance", 80000),
    (7, "Grace", "Finance", 110000),
    (8, "Helen", "Finance", 150000),
    (9, "Ian", "Finance", 95000)
]

columns = ["emp_id", "emp_name", "dept", "salary"]

df = spark.createDataFrame(data, schema=columns)
df.display()


In [0]:
df_avg = df.agg(avg("salary").alias("avg_sal"))
df_dept_max_sal = df.groupBy("dept").agg(max("salary").alias("dept_max_sal"))

df_join = df_dept_max_sal.join(df, "dept", "inner").join(df_avg)
df_final = df_join.filter((col("salary") > col("avg_sal")) & (col("salary") < col("dept_max_sal")))
display(df_final)


